# Hotel Review Classification — Inference (Kaggle)

Loads the sentiment and aspect models saved by **kaggle_train.ipynb** and
runs them on example sentences. This notebook does no data cleaning and no
training — it only loads two saved checkpoints and calls them.

## Before you run this

1. Run `kaggle_train.ipynb` first and save/commit a version, so Kaggle keeps
   `artifacts/sentiment` and `artifacts/aspects` as that notebook's Output.
2. In this notebook, Add Data → attach that Output as an input dataset.
3. Update `SENTIMENT_MODEL_DIR` and `ASPECT_MODEL_DIR` below to the exact
   path Kaggle mounts it at (shown in the Input panel on the right after you
   attach it, usually `/kaggle/input/<dataset-name>/artifacts/...`).
4. A GPU is optional here — a single sentence runs fast even on CPU.


In [ ]:
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
# Update these two paths to match where Kaggle mounted your attached dataset.
SENTIMENT_MODEL_DIR = Path("/kaggle/input/hrast-artifacts/artifacts/sentiment")
ASPECT_MODEL_DIR = Path("/kaggle/input/hrast-artifacts/artifacts/aspects")
MAX_LENGTH = 128


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("No GPU found. Using CPU, which is fine for single sentences.")


## Step 1: Define the labels

Must match the labels used during training.

In [ ]:
SENTIMENT_LABELS = ["negative", "neutral", "positive"]

ASPECT_COLUMNS = [
    "Clean", "Comfort", "Facilities/Amenities", "Location",
    "Restaurant (dinner)", "Staff", "View (Balcony)", "Breakfast", "Room",
    "Pool", "Beach", "Bathroom/Shower (toilet)", "Bar", "Bed", "Parking",
    "Noise", "Reception-checkin", "Lift", "Value for money", "Wi-Fi", "Generic",
]


## Step 2: Load both trained models

In [ ]:
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_DIR)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_DIR)
sentiment_model.to(device)
sentiment_model.eval()


In [ ]:
aspect_tokenizer = AutoTokenizer.from_pretrained(ASPECT_MODEL_DIR)
aspect_model = AutoModelForSequenceClassification.from_pretrained(ASPECT_MODEL_DIR)
aspect_model.to(device)
aspect_model.eval()


In [ ]:
with open(ASPECT_MODEL_DIR / "threshold.json") as f:
    threshold_info = json.load(f)
aspect_threshold = threshold_info["global_threshold"]
print("Using aspect threshold:", aspect_threshold)


## Step 3: Prediction functions

In [ ]:
def predict_sentiment(text):
    encoded = sentiment_tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = sentiment_model(**encoded)

    probabilities = torch.softmax(output.logits, dim=1)[0].cpu().tolist()
    scores = {}
    for i in range(len(SENTIMENT_LABELS)):
        scores[SENTIMENT_LABELS[i]] = round(probabilities[i], 4)

    predicted_id = int(torch.argmax(output.logits, dim=1)[0])
    predicted_label = SENTIMENT_LABELS[predicted_id]

    return predicted_label, scores


In [ ]:
def predict_aspects(text):
    encoded = aspect_tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = aspect_model(**encoded)

    probabilities = torch.sigmoid(output.logits)[0].cpu().tolist()

    detected_aspects = []
    for i in range(len(ASPECT_COLUMNS)):
        if probabilities[i] >= aspect_threshold:
            detected_aspects.append((ASPECT_COLUMNS[i], round(probabilities[i], 4)))

    detected_aspects.sort(key=lambda item: item[1], reverse=True)
    return detected_aspects


## Step 4: Try it out

Edit `example_sentences` to test your own review text.

In [ ]:
example_sentences = [
    "The staff were friendly and the room was spotless.",
    "The Wi-Fi was slow and the room was noisy.",
    "Breakfast was poor but the staff were excellent.",
    "The hotel is located three kilometres from the airport.",
]

for sentence in example_sentences:
    predicted_label, scores = predict_sentiment(sentence)
    aspects_found = predict_aspects(sentence)

    print("Review:", sentence)
    print("Sentiment:", predicted_label, scores)
    print("Aspects:", aspects_found)
    print()
